In [3]:
# ==========================================
# IMPORTS
# ==========================================
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import os
from tqdm import tqdm
from joblib import Parallel, delayed

# ==========================================
# PATHS
# ==========================================
BASE_PATH    = os.getcwd()
IMAGE_FOLDER = os.path.join(BASE_PATH, "images")
MASK_FOLDER  = os.path.join(BASE_PATH, "masks")
OUTPUT_CSV   = os.path.join(BASE_PATH, "feature_matrix.csv")
TOTAL_TILES  = 1340

print(f"Base path    : {BASE_PATH}")
print(f"Images exist : {os.path.exists(IMAGE_FOLDER)}")
print(f"Masks exist  : {os.path.exists(MASK_FOLDER)}")

# ==========================================
# COLOUR CLASS TABLE
# ==========================================
CLASS_COLOURS = {
    1: {"name": "Utility",       "rgb": (255, 18,  77)},
    2: {"name": "Water_Body_Pt", "rgb": (33,  107, 255)},
    3: {"name": "RCC_Building",  "rgb": (207, 110, 0)},
    4: {"name": "Utility_Poly",  "rgb": (0,   87,  128)},
    5: {"name": "Tiled",         "rgb": (233, 255, 64)},
    6: {"name": "Road",          "rgb": (255, 176, 184)},
    7: {"name": "Tin",           "rgb": (88,  0,   143)},
    8: {"name": "Water_Body",    "rgb": (64,  255, 249)},
}

TOLERANCE = 30

# ==========================================
# FEATURE NAMES (30)
# ==========================================
FEATURE_NAMES = [
    'R_mean',  'R_std',  'R_min',  'R_max',
    'G_mean',  'G_std',  'G_min',  'G_max',
    'B_mean',  'B_std',  'B_min',  'B_max',
    'Gray_mean','Gray_std','Gray_min','Gray_max',
    'Edge_mean','Edge_std',
    'Texture_contrast','Texture_energy',
    'Texture_homogeneity','Texture_correlation',
    'Area','Perimeter','Compactness',
    'Hist_R','Hist_G','Hist_B',
    'Saturation_mean','Brightness_mean'
]

# ==========================================
# BUILD COLUMN NAMES (250 total)
# ==========================================
COLUMNS = ['Tile_Name', 'Classes_Present']

for cid, cinfo in CLASS_COLOURS.items():
    COLUMNS.append(cinfo['name'] + '_class')

for cid, cinfo in CLASS_COLOURS.items():
    for fname in FEATURE_NAMES:
        COLUMNS.append(cinfo['name'] + '_' + fname)

print(f"Total columns: {len(COLUMNS)}")  # Must print 250

# ==========================================
# EXTRACT 30 FEATURES FROM PIXEL REGION
# ==========================================
def extract_features(raw_array, pixel_mask):
    pixels = raw_array[pixel_mask]

    if len(pixels) == 0:
        return [0.0] * 30

    R = pixels[:, 0].astype(float)
    G = pixels[:, 1].astype(float)
    B = pixels[:, 2].astype(float)

    gray = 0.299 * R + 0.587 * G + 0.114 * B

    # COLOUR FEATURES (12)
    R_mean = float(np.mean(R));  R_std = float(np.std(R))
    R_min  = float(np.min(R));   R_max = float(np.max(R))
    G_mean = float(np.mean(G));  G_std = float(np.std(G))
    G_min  = float(np.min(G));   G_max = float(np.max(G))
    B_mean = float(np.mean(B));  B_std = float(np.std(B))
    B_min  = float(np.min(B));   B_max = float(np.max(B))

    # GRAYSCALE FEATURES (4)
    Gray_mean = float(np.mean(gray))
    Gray_std  = float(np.std(gray))
    Gray_min  = float(np.min(gray))
    Gray_max  = float(np.max(gray))

    # EDGE FEATURES (2)
    if len(gray) > 1:
        diff      = np.abs(np.diff(gray))
        Edge_mean = float(np.mean(diff))
        Edge_std  = float(np.std(diff))
    else:
        Edge_mean = 0.0
        Edge_std  = 0.0

    # TEXTURE FEATURES (4)
    Texture_contrast    = float(np.var(gray))
    Texture_energy      = float(np.sum((gray / 255.0) ** 2) / len(gray))
    Texture_homogeneity = float(np.mean(1.0 / (1.0 + np.abs(gray - Gray_mean))))

    if len(R) > 1 and float(np.std(R)) > 0 and float(np.std(G)) > 0:
        Texture_correlation = float(np.corrcoef(R, G)[0, 1])
        if np.isnan(Texture_correlation):
            Texture_correlation = 0.0
    else:
        Texture_correlation = 0.0

    # SHAPE FEATURES (3)
    Area        = float(len(pixels))
    Perimeter   = float(np.sqrt(len(pixels)))
    Compactness = float(Area / (Perimeter ** 2)) if Perimeter > 0 else 0.0

    # HISTOGRAM FEATURES (3)
    Hist_R = float(np.percentile(R, 75) - np.percentile(R, 25))
    Hist_G = float(np.percentile(G, 75) - np.percentile(G, 25))
    Hist_B = float(np.percentile(B, 75) - np.percentile(B, 25))

    # HSV FEATURES (2)
    r_val = min(int(R_mean), 255)
    g_val = min(int(G_mean), 255)
    b_val = min(int(B_mean), 255)
    mean_pixel      = np.array([[[r_val, g_val, b_val]]], dtype=np.uint8)
    hsv             = cv2.cvtColor(mean_pixel, cv2.COLOR_RGB2HSV)
    Saturation_mean = float(hsv[0, 0, 1])
    Brightness_mean = float(hsv[0, 0, 2])

    features = [
        R_mean, R_std, R_min, R_max,
        G_mean, G_std, G_min, G_max,
        B_mean, B_std, B_min, B_max,
        Gray_mean, Gray_std, Gray_min, Gray_max,
        Edge_mean, Edge_std,
        Texture_contrast, Texture_energy,
        Texture_homogeneity, Texture_correlation,
        Area, Perimeter, Compactness,
        Hist_R, Hist_G, Hist_B,
        Saturation_mean, Brightness_mean
    ]

    return [round(f, 4) for f in features]

# ==========================================
# PROCESS SINGLE TILE (called by joblib)
# ==========================================
def process_tile(i):
    image_path = os.path.join(IMAGE_FOLDER, f"image_{i}.png")
    mask_path  = os.path.join(MASK_FOLDER,  f"mask_{i}.png")

    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        return None

    raw_img  = np.array(Image.open(image_path).convert("RGB"))
    mask_img = np.array(Image.open(mask_path).convert("RGB"))

    class_presence = {}
    for class_id, class_info in CLASS_COLOURS.items():
        tr, tg, tb = class_info['rgb']
        pixel_mask = (
            (np.abs(mask_img[:, :, 0].astype(int) - tr) <= TOLERANCE) &
            (np.abs(mask_img[:, :, 1].astype(int) - tg) <= TOLERANCE) &
            (np.abs(mask_img[:, :, 2].astype(int) - tb) <= TOLERANCE)
        )
        class_presence[class_id] = pixel_mask

    row = {}
    row['Tile_Name'] = f"image_{i}"

    present_classes = [
        str(cid) for cid, pmask in class_presence.items()
        if pmask.sum() > 0
    ]
    row['Classes_Present'] = ','.join(present_classes) if present_classes else '0'

    for class_id, class_info in CLASS_COLOURS.items():
        col_name      = class_info['name'] + '_class'
        row[col_name] = class_id if class_presence[class_id].sum() > 0 else 0

    for class_id, class_info in CLASS_COLOURS.items():
        features = extract_features(raw_img, class_presence[class_id])
        for fname, fval in zip(FEATURE_NAMES, features):
            row[class_info['name'] + '_' + fname] = fval

    return row

# ==========================================
# FULL RUN WITH 8 CORES
# ==========================================
print("Starting full run on 1340 tiles using 8 cores...")

results = Parallel(n_jobs=8, verbose=1)(
    delayed(process_tile)(i) for i in range(TOTAL_TILES)
)

# Remove None results (missing tiles)
results = [r for r in results if r is not None]

# Save to CSV
df = pd.DataFrame(results, columns=COLUMNS)

# Sort by tile number so output is in order
df['tile_num'] = df['Tile_Name'].str.extract(r'(\d+)').astype(int)
df = df.sort_values('tile_num').drop('tile_num', axis=1).reset_index(drop=True)

df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Feature matrix saved → {OUTPUT_CSV}")
print(f"   Rows    : {len(df)}")
print(f"   Columns : {len(df.columns)}")
print(f"\nPreview:")
print(df[['Tile_Name', 'Classes_Present'] +
         [c for c in COLUMNS if '_class' in c]].head())

Base path    : C:\Users\nsury\Desktop\Drone_Project
Images exist : True
Masks exist  : True
Total columns: 250
Starting full run on 1340 tiles using 8 cores...


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 1200 tasks      | elapsed:    4.7s
[Parallel(n_jobs=8)]: Done 1340 out of 1340 | elapsed:    5.1s finished



✅ Feature matrix saved → C:\Users\nsury\Desktop\Drone_Project\feature_matrix.csv
   Rows    : 1340
   Columns : 250

Preview:
  Tile_Name Classes_Present  Utility_class  Water_Body_Pt_class  \
0   image_0           3,6,7              0                    0   
1   image_1           3,6,7              0                    0   
2   image_2             3,6              0                    0   
3   image_3             6,8              0                    0   
4   image_4               8              0                    0   

   RCC_Building_class  Utility_Poly_class  Tiled_class  Road_class  Tin_class  \
0                   3                   0            0           6          7   
1                   3                   0            0           6          7   
2                   3                   0            0           6          0   
3                   0                   0            0           6          0   
4                   0                   0            0           

In [4]:
OUTPUT_CSV = os.path.join(os.path.expanduser("~"), "Desktop", "feature_matrix_t30.csv")

df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Saved → {OUTPUT_CSV}")
print(f"   Rows    : {len(df)}")
print(f"   Columns : {len(df.columns)}")

✅ Saved → C:\Users\nsury\Desktop\feature_matrix_t30.csv
   Rows    : 1340
   Columns : 250
